In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ntphiep/viT5_tst_coarse")
model = AutoModelForSeq2SeqLM.from_pretrained("ntphiep/viT5_tst_coarse")

/usr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.

In [ ]:
import sagemaker
import boto3
import joblib

sm = boto3.client("sagemaker")
sm_session = sagemaker.Session()

In [16]:
from sagemaker.huggingface import HuggingFaceModel

huggingface_model = HuggingFaceModel(
    model_data=None,  # không cần nếu load từ hub
    transformers_version='4.37.0',
    pytorch_version='2.1.0',
    py_version='py310',
    env={
        'HF_MODEL_ID': 'ntphiep/viT5_tst_coarse',  # model trên HF của mày
        'HF_TASK': 'text2text-generation'
    },
    role='arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510'
)


In [17]:
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

-------!

In [23]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

endpoint_name = "huggingface-pytorch-inference-2025-07-06-10-55-21-255"  # ví dụ: vit5-paraphrase-endpoint
payload = {
  "inputs": "Hôm nay là một ngày tuyệt vời để đi chơi."
}

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",  # ✅ đúng MIME type
    Body=json.dumps(payload)         # ✅ serialize chuẩn
)

print(json.loads(response["Body"].read().decode()))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8   "inputs": "Hôm nay là một ngày tuyệt vời để đi chơi."                                     │
│    9 }                                                                                           │
│   10                                                                                             │
│ ❱ 11 response = runtime.invoke_endpoint(                                                         │
│   12 │   EndpointName=endpoint_name,                                                             │
│   13 │   ContentType="application/json",  # ✅ đúng MIME type                                    │
│   14 │   Body=json.dumps(payload)         # ✅ serialize chuẩn                                   │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/client.py:595 in _api_call                            │
│                                                                                                  │
│    592 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    593 │   │   │   │   )                                                                         │
│    594 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  595 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    596 │   │                                                                                     │
│    597 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    598                                                                                           │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/context.py:123 in wrapper                             │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/client.py:1058 in _make_api_call                      │
│                                                                                                  │
│   1055 │   │   │   │   "Code"                                                                    │
│   1056 │   │   │   )                                                                             │
│   1057 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1058 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1059 │   │   else:                                                                             │
│   1060 │   │   │   return parsed_response                                                        │
│   1061                                                       

In [ ]:
from sagemaker.huggingface import HuggingFaceModel
import sagemaker

role = "arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510"

huggingface_model = HuggingFaceModel(
    model_data="s3://hiep-delta-bk/models/model_coarse.tar.gz",
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
    entry_point="inference.py",  # 👈 custom handler của mày
    source_dir="./model_coarse",  # 👈 nơi chứa inference.py và model dir
)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="vit5-tst-coarse-endpoint"
)


sagemaker.config INFO - Not applying SDK defaults from location: /home/hiep/.config/kdedefaults/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/hiep/.config/sagemaker/config.yaml


In [29]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName="vit5-tst-coarse-endpoint",
    ContentType="application/json",
    Body=json.dumps({"inputs": "Tôi rất yêu thích thời tiết hôm nay."})
)

print(response["Body"].read().decode())


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3                                                                                             │
│    4 runtime = boto3.client("sagemaker-runtime")                                                 │
│    5                                                                                             │
│ ❱  6 response = runtime.invoke_endpoint(                                                         │
│    7 │   EndpointName="vit5-tst-coarse-endpoint",                                                │
│    8 │   ContentType="application/json",                                                         │
│    9 │   Body=json.dumps({"inputs": "Tôi rất yêu thích thời tiết hôm nay."})                     │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/client.py:595 in _api_call                            │
│                                                                                                  │
│    592 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    593 │   │   │   │   )                                                                         │
│    594 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  595 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    596 │   │                                                                                     │
│    597 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    598                                                                                           │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/context.py:123 in wrapper                             │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /usr/lib/python3.13/site-packages/botocore/client.py:1058 in _make_api_call                      │
│                                                                                                  │
│   1055 │   │   │   │   "Code"                                                                    │
│   1056 │   │   │   )                                                                             │
│   1057 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1058 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1059 │   │   else:                                                                             │
│   1060 │   │   │   return parsed_response                                                        │
│   1061                                                     